# 하이브리드 검색

In [1]:
# pip install rank_bm25 #키워드 기반의 랭킹 알고리즘
# pip install langchain-classic

In [2]:
# 0. 필요한 라이브러리 import
from langchain_classic.retrievers import EnsembleRetriever

from langchain_community.retrievers import BM25Retriever # 키워드 기반 검색 라이브러리
from langchain_community.vectorstores import FAISS # 벡터 유사도 검색 라이브러리
from langchain_openai import OpenAIEmbeddings # OpenAI 임베딩 모델

C:\Users\qkrru\AppData\Local\Temp\ipykernel_11852\3852043606.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever # 키워드 기반 검색 라이브러리


In [3]:
# 1. 환경 변수로부터 키값 가져오기
import os
from dotenv import load_dotenv

# 환경 변수 로드 및 OpenAI 클라이언트 설정
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [4]:
# 2. 첫 번째 데이터셋 준비 (강아지 성격 특성)
doc_list_1 = [
    "프렌치 불독: 사교적이고 친근한 성격을 가지고 있으며, 조용하고 집에서 지내기에 적합 합니다",
    "비글: 호기심이 많고, 에너지가 넘치며, 사냥 본능이 강합니다. ",
    "독일 셰퍼드: 용감하고 지능적이며, 충성심이 강합니다",
    "포메라니안: 활발하고 호기심이 많으며, 주인에게 매우 애정적입니다",
    "치와와: 작지만 용감하고, 주인에게 깊은 애정을 보입니다",
    "보더 콜리:	매우 지능적이고 학습 능력이 뛰어나며, 에너지가 많아 많은 운동이 필요합니다 "
]

# 3. 두 번째 데이터셋 준비 (강아지 관리 및 특이사항)
doc_list_2 = [
    "프렌치 불독: 열에 약하므로 주의가 필요합니다",
    "비글: 가족과 잘 지내며, 아이들과 노는 것을 좋아합니다.",
    "독일 셰퍼드: 경찰견이나 구조견으로 많이 활용되며, 적절한 훈련과 운동이 필요합니다.",
    "포메라니안: 털이 풍성하므로 정기적인 그루밍이 필요합니다.",
    "치와와: 다른 동물이나 낯선 사람에게는 조심스러울 수 있습니다.",
    "보더 콜리: 목축견으로서의 본능이 강하며, 다양한 트릭과 명령을 쉽게 배울 수 있습니다."
]

In [ ]:
# 4. OpenAI 임베딩 모델 초기화 (벡터 검색용)
embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=api_key
)

# 5. BM25 Retriever 초기화 (키워드 기반 검색)
bm25_retriever = BM25Retriever.from_texts(
    doc_list_1,
    metadatas=[{"source": 1}] * len(doc_list_1) # 첫 번째 데이터셋 메타데이터
)
bm25_retriever.k = 2 # 상위 2개 문서 반

# 6. FAISS Retriever 초기화 (벡터 유사도 기반 검색)
faiss_vectorstore = FAISS.from_texts(
    doc_list_2,
    embedding,
    metadatas=[{"source": 2}] * len(doc_list_2) # 두 번째 데이터셋 메타데이
)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# 7. Emsemble Retriever 초기화 (두 검색 방법을 조합)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5] # 동일 가중치 부여
)

In [8]:
# 8. 첫 번째 질문 테스트: '충성심이 강한 강아지는?'
query = "충성심이 강한 강아지는?"

bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)
ensemble_result = ensemble_retriever.invoke(query)

# 첫 번째 질문 결과 출력
print("[BM25 Retriever - 키워드 검색]\n", bm25_result, end="\n\n")
print("[FAISS Retriever - 벡터 검색]\n", faiss_result, end="\n\n")
print("[Ensemble Retriever - 조합 검색]\n", ensemble_result, end="\n\n")

[BM25 Retriever - 키워드 검색]
 [Document(metadata={'source': 1}, page_content='독일 셰퍼드: 용감하고 지능적이며, 충성심이 강합니다'), Document(metadata={'source': 1}, page_content='보더 콜리:\t매우 지능적이고 학습 능력이 뛰어나며, 에너지가 많아 많은 운동이 필요합니다 ')]

[FAISS Retriever - 벡터 검색]
 [Document(id='28092059-f23e-4820-83dc-a775912a37b1', metadata={'source': 2}, page_content='치와와: 다른 동물이나 낯선 사람에게는 조심스러울 수 있습니다.'), Document(id='d076b558-2216-49cb-8a4a-aaaa67ff6f79', metadata={'source': 2}, page_content='포메라니안: 털이 풍성하므로 정기적인 그루밍이 필요합니다.')]

[Ensemble Retriever - 조합 검색]
 [Document(metadata={'source': 1}, page_content='독일 셰퍼드: 용감하고 지능적이며, 충성심이 강합니다'), Document(id='28092059-f23e-4820-83dc-a775912a37b1', metadata={'source': 2}, page_content='치와와: 다른 동물이나 낯선 사람에게는 조심스러울 수 있습니다.'), Document(metadata={'source': 1}, page_content='보더 콜리:\t매우 지능적이고 학습 능력이 뛰어나며, 에너지가 많아 많은 운동이 필요합니다 '), Document(id='d076b558-2216-49cb-8a4a-aaaa67ff6f79', metadata={'source': 2}, page_content='포메라니안: 털이 풍성하므로 정기적인 그루밍이 필요합니다.')]



In [9]:
# 9. 두 번째 질문 테스트: "지능적인 강아지는?"

query = "지능적인 강아지는?"

bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)
ensemble_result = ensemble_retriever.invoke(query)

# 두 번째 질문 결과 출력

print("[BM25 Retriever - 키워드 검색]\n", bm25_result, end="\n\n")
print("[FAISS Retriever - 벡터 검색]\n", faiss_result, end="\n\n")
print("[Ensemble Retriever - 조합 검색]\n", ensemble_result, end="\n\n")

[BM25 Retriever - 키워드 검색]
 [Document(metadata={'source': 1}, page_content='보더 콜리:\t매우 지능적이고 학습 능력이 뛰어나며, 에너지가 많아 많은 운동이 필요합니다 '), Document(metadata={'source': 1}, page_content='치와와: 작지만 용감하고, 주인에게 깊은 애정을 보입니다')]

[FAISS Retriever - 벡터 검색]
 [Document(id='28092059-f23e-4820-83dc-a775912a37b1', metadata={'source': 2}, page_content='치와와: 다른 동물이나 낯선 사람에게는 조심스러울 수 있습니다.'), Document(id='243c1eff-34fb-4d35-9687-0117e22d0b70', metadata={'source': 2}, page_content='보더 콜리: 목축견으로서의 본능이 강하며, 다양한 트릭과 명령을 쉽게 배울 수 있습니다.')]

[Ensemble Retriever - 조합 검색]
 [Document(metadata={'source': 1}, page_content='보더 콜리:\t매우 지능적이고 학습 능력이 뛰어나며, 에너지가 많아 많은 운동이 필요합니다 '), Document(id='28092059-f23e-4820-83dc-a775912a37b1', metadata={'source': 2}, page_content='치와와: 다른 동물이나 낯선 사람에게는 조심스러울 수 있습니다.'), Document(metadata={'source': 1}, page_content='치와와: 작지만 용감하고, 주인에게 깊은 애정을 보입니다'), Document(id='243c1eff-34fb-4d35-9687-0117e22d0b70', metadata={'source': 2}, page_content='보더 콜리: 목축견으로서의 본능이 강하며, 다양한 트릭과 명령을 쉽게 배울 수 있습니다.')]